In [2]:
%load_ext autoreload
%autoreload 2

import os
print(os.getcwd())
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.optim import Adam,AdamW
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import pickle

from train_re import AdaptiveTAPEandDiffusion2, alternate_training_earlyStop, evaluation, adaptive_stage_domain9
from utils import simdatset, reproducibility, calculate_evaluation_metrics

batch_size = 256
reproducibility(2023)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/disk1/user/liaoshuilin/project/35.TAPE_EXO/assay_diffusion


# model 

In [ ]:
# var_values = [0.3, 0.6, 0.9]
# r_values = ["r1", "r2", "r3"] # 1000/2000/5000
var_values = [0.3]
r_values = ["r1"]

for var in var_values:
    for r in r_values:
        out_pth = "../result/model_variFeats/var" + str(var) + "_" + r + "_"
        print(out_pth)

        with open(f'../result/data_variFeats/Stim_data_' + str(var) + '.pkl', 'rb') as file:
            loaded_data = pickle.load(file)
        GTE_x_train = loaded_data['GTE_x_' + r + '_train']
        GTE_x_val = loaded_data['GTE_x_' + r + '_val']
        GTE_x_test = loaded_data['GTE_x_' + r + '_test']
        GTE_y_train = loaded_data['GTE_y_' + r + '_train']
        GTE_y_val = loaded_data['GTE_y_' + r + '_val']
        GTE_y_test = loaded_data['GTE_y_' + r + '_test']
        HPA_x = loaded_data['HPA_x_' + r]
        HPA_y = loaded_data['HPA_y_' + r ]

        ## stage1 model
        model = AdaptiveTAPEandDiffusion2(GTE_x_train.shape[1], GTE_y_train.shape[1],2, T=2000).to(device)
        optimizer_main = AdamW([
            
            {'params': model.encoder.parameters()},
            {'params': model.predictor.parameters()},
            {'params': model.decoder.parameters()}], lr=1e-4)
        optimizer_diffusion =AdamW(model.ref_creator.parameters(), lr=1e-3)
        optimizer_all = AdamW(model.parameters(),1e-4)
        epochs_main = 400 
        epochs_diffusion = 1500

        train_loader = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(simdatset(GTE_x_val, GTE_y_val), batch_size=batch_size, shuffle=False)
        model, main_loss, diffloss = alternate_training_earlyStop(model, train_loader, val_loader, optimizer_main, optimizer_diffusion, 
                                                        epochs_main, epochs_diffusion, device='cpu',
                                                        patience=5, early_stop_start=400, early_stop_interval=50)
        torch.save(model, out_pth + "model_stage1.pth")

        train_loader2 = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=False)
        x_recon_tr, f_tr, z_tr  = evaluation(train_loader2, model, device=device)
        sigmatrix = np.linalg.pinv(f_tr) @ x_recon_tr 
        pd.DataFrame(sigmatrix).to_csv(out_pth + 'sigmatrix.csv', index=True, header=True)

        ## DADA
        x_recon_HPA, f_HPA, z_HPA, model2 = adaptive_stage_domain9(x=HPA_x, model_name=out_pth + "model_stage1", mode = 'overall5', steps=10, max_iter=40, device=device, sigmatrix = sigmatrix)
        pd.DataFrame(x_recon_HPA).to_csv(out_pth + 'HPA_DADA_x_recon.csv', index=False, header=False)
        pd.DataFrame(f_HPA).to_csv(out_pth + 'HPA_DADA_y.csv', index=False, header=False)
        rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x, out_pth = out_pth + "HPA_DADA_x_")
        print(f"DADA\nRMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
        rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y, out_pth = out_pth + "HPA_DADA_y_")
        print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")
        torch.save(model2, out_pth + "model_stage2.pth")

# write csv

In [4]:
var_values = [0.3, 0.6, 0.9]
r_values = ["r1", "r2", "r3"]

for var in var_values:
    for r in r_values:
        out_pth = "/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_variFeats/var" + str(var) + "_" + r + "_"
        print(out_pth)

        with open(f'../result/data_variFeats/Stim_data_' + str(var) + '.pkl', 'rb') as file:
            loaded_data = pickle.load(file)
        
        GTE_x = loaded_data['GTE_x_' + r]

        GTE_x_train = loaded_data['GTE_x_' + r + '_train']
        GTE_x_val = loaded_data['GTE_x_' + r + '_val']
        GTE_x_test = loaded_data['GTE_x_' + r + '_test']
        GTE_y_train = loaded_data['GTE_y_' + r + '_train']
        GTE_y_val = loaded_data['GTE_y_' + r + '_val']
        GTE_y_test = loaded_data['GTE_y_' + r + '_test']
        HPA_x = loaded_data['HPA_x_' + r]
        HPA_y = loaded_data['HPA_y_' + r ]

        x_recon_HPA = pd.read_csv(out_pth + 'HPA_DADA_x_recon.csv', header=None)
        f_HPA =  pd.read_csv(out_pth + 'HPA_DADA_y.csv', header=None)

        rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x)
        print(f"DADA\nRMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
        rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y)
        print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

        GTE_x_mean=np.mean(GTE_x, axis=0)
        print(GTE_x_mean.shape)
        HPA_x_mean=np.mean(HPA_x, axis=0)
        print(HPA_x_mean.shape)

        # pd.DataFrame(GTE_x_mean).to_csv(out_pth + 'GTE_x_mean.csv', index=False, header=False)
        # pd.DataFrame(HPA_x_mean).to_csv(out_pth + 'HPA_x_mean.csv', index=False, header=False)

/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_variFeats/var0.3_r1_
DADA
RMSE of x: 0.0408
PCC of x: 0.9638
MAE of x: 0.0291
CCC of x: 0.9453
RMSE of test frac: 0.0406
PCC of frac: 0.7174
MAE of frac: 0.0283
CCC of frac: 0.6729
(1000,)
(1000,)
/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_variFeats/var0.3_r2_
DADA
RMSE of x: 0.0333
PCC of x: 0.9638
MAE of x: 0.0212
CCC of x: 0.9558
RMSE of test frac: 0.0391
PCC of frac: 0.7329
MAE of frac: 0.0269
CCC of frac: 0.6966
(2000,)
(2000,)
/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_variFeats/var0.3_r3_
DADA
RMSE of x: 0.0338
PCC of x: 0.9627
MAE of x: 0.0226
CCC of x: 0.9483
RMSE of test frac: 0.0375
PCC of frac: 0.7614
MAE of frac: 0.0263
CCC of frac: 0.7196
(5000,)
(5000,)
/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_variFeats/var0.6_r1_
DADA
RMSE of x: 0.0326
PCC of x: 0.9789
MAE of x: 0.0222
CCC of x: 0.9709
RMSE of test frac: 0.0346
PCC of frac: 0.8099
MAE of frac: 0.0233
CCC of frac: 0.7863
